In [1]:
# python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout,  llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
import coffea.util

In [2]:
client = scaleout.make_dask_client("tls://localhost:8786")
client

Connection method: Direct,
Dashboard: /user/maria.jose@cern.ch/proxy/8787/status,
Comm: tls://192.168.161.128:8786,Workers: 0
Dashboard: /user/maria.jose@cern.ch/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [3]:
import yaml
yaml_file_path = '../../configs/ntuples/signal_2mu2e_v10.yaml'
# Open and read the YAML file
with open(yaml_file_path, 'r') as file:
    data = yaml.safe_load(file)
signals_all = list(data["llpNanoAOD_v2"]["samples"].keys())
# print(signals_all)
for x in signals_all:
    print("'"+x+"',")

'2Mu2E_100GeV_0p25GeV_0p02mm',
'2Mu2E_100GeV_0p25GeV_0p2mm',
'2Mu2E_100GeV_0p25GeV_2p0mm',
'2Mu2E_100GeV_0p25GeV_10p0mm',
'2Mu2E_100GeV_0p25GeV_20p0mm',
'2Mu2E_100GeV_1p2GeV_0p096mm',
'2Mu2E_100GeV_1p2GeV_0p96mm',
'2Mu2E_100GeV_1p2GeV_9p6mm',
'2Mu2E_100GeV_1p2GeV_48p0mm',
'2Mu2E_100GeV_1p2GeV_96p0mm',
'2Mu2E_100GeV_5p0GeV_0p4mm',
'2Mu2E_100GeV_5p0GeV_4p0mm',
'2Mu2E_100GeV_5p0GeV_40p0mm',
'2Mu2E_100GeV_5p0GeV_200mm',
'2Mu2E_100GeV_5p0GeV_400mm',
'2Mu2E_150GeV_0p25GeV_0p013mm',
'2Mu2E_150GeV_0p25GeV_0p13mm',
'2Mu2E_150GeV_0p25GeV_1p3mm',
'2Mu2E_150GeV_0p25GeV_6p7mm',
'2Mu2E_150GeV_0p25GeV_13p0mm',
'2Mu2E_150GeV_1p2GeV_0p064mm',
'2Mu2E_150GeV_1p2GeV_0p64mm',
'2Mu2E_150GeV_1p2GeV_6p4mm',
'2Mu2E_150GeV_1p2GeV_32p0mm',
'2Mu2E_150GeV_1p2GeV_64p0mm',
'2Mu2E_150GeV_5p0GeV_0p27mm',
'2Mu2E_150GeV_5p0GeV_2p7mm',
'2Mu2E_150GeV_5p0GeV_27p0mm',
'2Mu2E_150GeV_5p0GeV_130p0mm',
'2Mu2E_150GeV_5p0GeV_270p0mm',
'2Mu2E_200GeV_0p25GeV_0p01mm',
'2Mu2E_200GeV_0p25GeV_0p1mm',
'2Mu2E_200GeV_0p25GeV_1p0mm',
'2Mu2

In [4]:
runner = processor.Runner(
    #executor=processor.IterativeExecutor(),
    executor=processor.DaskExecutor(client=client),
    # schema=NanoAODSchema,
    schema = llpnanoaodschema.LLPNanoAODSchema,
    #maxchunks=1,
    skipbadfiles=True,
)
channelname="base_ljObjCut_ljIso_2lj"
hist_collections = "displacement_base"
max_files_data = -1
max_files_bg = 1000

p = sidm_processor.SidmProcessor(
    [channelname],
    [
     hist_collections
    ],

)

In [5]:

bgs= [
    
     # "TTJets",
     #     "QCD_Pt1000",
     #   "QCD_Pt800To1000",

      # "QCD_Pt600To800",
       # "QCD_Pt470To600",
       #  "QCD_Pt300To470",
     
      # "QCD_Pt170To300",
      #    "QCD_Pt120To170",
      #   "QCD_Pt80To120",
      #    "QCD_Pt50To80",
      #  "QCD_Pt30To50",
      #  "QCD_Pt20To30", #error
      #    "QCD_Pt15To20", #error
 
 
      # "DYJetsToMuMu_M10to50",
       "DYJetsToMuMu_M50",
     
]



In [6]:
import os
dir_path = channelname + hist_collections
os.makedirs(dir_path, exist_ok=True)

In [7]:
# # signals_all = [
   
# # '2Mu2E_500GeV_0p25GeV_0p04mm',
# # '2Mu2E_500GeV_0p25GeV_2p0mm',
# # '2Mu2E_500GeV_5p0GeV_0p8mm',
# # '2Mu2E_500GeV_5p0GeV_40p0mm',

# ]
# fileset = utilities.make_fileset(signals_all, "llpNanoAOD_v2", max_files=max_files_data, location_cfg="signal_2mu2e_v10.yaml")
# output_signal = runner.run(fileset, treename="Events", processor_instance=p)
# coffea.util.save(output_signal, f"{dir_path}/output_signal.coffea"  )


In [8]:
for x in bgs:
   print(x)
   fileset = utilities.make_fileset([x], "skimmed_llpNanoAOD_v2", location_cfg="backgrounds.yaml", max_files=max_files_bg)
   # print(fileset)
   output = runner.run(fileset, treename="Events", processor_instance=p)
   coffea.util.save(output, f"{dir_path}/output_{x}.coffea"  )

DYJetsToMuMu_M50


Output()

Output()

DYJetsToMuMu_M50 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [9]:
 # coffea.util.save(output_signal, "output_" + x + channelname+  ".coffea"  )